# VGG16 vs ResNet18 Result Analysis

This notebook reads the saved deep-learning training outputs and creates comparison figures. It does not retrain models.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220

ROOT = Path.cwd()
if ROOT.name.lower() in {"training", "analysis"}:
    PROJECT_ROOT = ROOT.parents[1]
else:
    PROJECT_ROOT = ROOT

RESULT_DIR = PROJECT_ROOT / "artifacts" / "deep_learning" / "training"
FIG_DIR = RESULT_DIR / "analysis_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["resnet18", "vgg16"]
DISPLAY_NAMES = {"resnet18": "ResNet18", "vgg16": "VGG16"}

RESULT_DIR, FIG_DIR

## Load Saved Results

In [ ]:
histories = {}
per_class = {}
predictions = {}
confusions = {}
metrics = {}

for model in MODELS:
    model_dir = RESULT_DIR / model
    histories[model] = pd.read_csv(model_dir / f"history_{model}_v5.csv")
    per_class[model] = pd.read_csv(model_dir / f"per_class_metrics_{model}_v5.csv")
    predictions[model] = pd.read_csv(model_dir / f"test_predictions_{model}_v5.csv")
    confusions[model] = pd.read_csv(model_dir / f"confusion_matrix_{model}_v5.csv", header=None)
    with open(model_dir / f"metrics_{model}_v5.json", "r", encoding="utf-8") as f:
        metrics[model] = json.load(f)

comparison = pd.read_csv(RESULT_DIR / "model_comparison_v5.csv")
class_counts = pd.read_csv(RESULT_DIR / "balanced_class_counts_v5.csv")
classes = class_counts["class"].tolist()

comparison

## Figure 1: Class Counts Before and After Downsampling

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(class_counts))
width = 0.38
ax.bar(x - width / 2, class_counts["original"], width, label="Original", color="#8da0cb")
ax.bar(x + width / 2, class_counts["used"], width, label="Used", color="#66c2a5")
ax.set_xticks(x, class_counts["class"], rotation=35, ha="right")
ax.set_ylabel("Number of images")
ax.set_title("Class Distribution Before and After Downsampling")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "01_class_distribution_downsampling.png")
plt.show()

## Figure 2: Training and Validation Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, model in zip(axes, MODELS):
    h = histories[model]
    ax.plot(h["epoch"], h["train_loss"], marker="o", label="Train loss")
    ax.plot(h["epoch"], h["val_loss"], marker="o", label="Validation loss")
    ax.set_title(f"{DISPLAY_NAMES[model]} Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "02_loss_curves_by_model.png")
plt.show()

## Figure 3: Training and Validation Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, model in zip(axes, MODELS):
    h = histories[model]
    ax.plot(h["epoch"], h["train_accuracy"] * 100, marker="o", label="Train accuracy")
    ax.plot(h["epoch"], h["val_accuracy"] * 100, marker="o", label="Validation accuracy")
    best_epoch = int(metrics[model]["best_epoch"])
    ax.axvline(best_epoch, color="gray", linestyle="--", alpha=0.6, label="Best epoch")
    ax.set_title(f"{DISPLAY_NAMES[model]} Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "03_accuracy_curves_by_model.png")
plt.show()

## Figure 4: Overall Test Performance Comparison

In [ ]:
plot_df = comparison.copy()
plot_df["model"] = plot_df["model"].map(DISPLAY_NAMES)
score_cols = ["test_accuracy", "macro_f1", "weighted_f1"]
long_scores = plot_df.melt(id_vars="model", value_vars=score_cols, var_name="metric", value_name="score")

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=long_scores, x="model", y="score", hue="metric", ax=ax, palette="Set2")
ax.set_ylim(0.9, 1.0)
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_title("Overall Test Performance")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "04_overall_test_performance.png")
plt.show()

## Figure 5: Training Time Comparison

In [ ]:
time_df = comparison[["model", "training_seconds"]].copy()
time_df["model"] = time_df["model"].map(DISPLAY_NAMES)
time_df["training_minutes"] = time_df["training_seconds"] / 60

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=time_df, x="model", y="training_minutes", ax=ax, palette=["#66c2a5", "#fc8d62"])
ax.set_xlabel("")
ax.set_ylabel("Training time (minutes)")
ax.set_title("Training Time Comparison")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f min", fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_training_time_comparison.png")
plt.show()

## Figure 6: Per-Class Accuracy Comparison

In [ ]:
rows = []
for model in MODELS:
    df = per_class[model].copy()
    df["model"] = DISPLAY_NAMES[model]
    rows.append(df)
per_class_all = pd.concat(rows, ignore_index=True)
per_class_all["accuracy_pct"] = per_class_all["accuracy"] * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=per_class_all, x="class", y="accuracy_pct", hue="model", ax=ax, palette="Set2")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_ylim(80, 100)
ax.set_xlabel("")
ax.set_ylabel("Per-class accuracy (%)")
ax.set_title("Per-Class Accuracy: ResNet18 vs VGG16")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_per_class_accuracy_comparison.png")
plt.show()

## Figure 7: Per-Class F1-Score Comparison

In [ ]:
per_class_all["f1_pct"] = per_class_all["f1"] * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=per_class_all, x="class", y="f1_pct", hue="model", marker="o", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_ylim(80, 100)
ax.set_xlabel("")
ax.set_ylabel("F1-score (%)")
ax.set_title("Per-Class F1-Score Comparison")
fig.tight_layout()
fig.savefig(FIG_DIR / "07_per_class_f1_comparison.png")
plt.show()

## Figure 8: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, model in zip(axes, MODELS):
    cm = confusions[model]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(f"{DISPLAY_NAMES[model]} Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
fig.tight_layout()
fig.savefig(FIG_DIR / "08_confusion_matrices_side_by_side.png")
plt.show()

## Figure 9: Error Counts by Class

In [ ]:
error_rows = []
for model in MODELS:
    df = predictions[model].copy()
    errors = df[df["correct"] == 0].groupby("true_label").size().reindex(classes, fill_value=0)
    for class_name, count in errors.items():
        error_rows.append({"model": DISPLAY_NAMES[model], "class": class_name, "errors": count})
error_df = pd.DataFrame(error_rows)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=error_df, x="class", y="errors", hue="model", ax=ax, palette="Set2")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_xlabel("")
ax.set_ylabel("Number of test errors")
ax.set_title("Test Error Counts by True Class")
fig.tight_layout()
fig.savefig(FIG_DIR / "09_error_counts_by_class.png")
plt.show()

## Figure 10: Error Overlap Between Models

In [ ]:
r = predictions["resnet18"][["path", "correct"]].rename(columns={"correct": "resnet18_correct"})
v = predictions["vgg16"][["path", "correct"]].rename(columns={"correct": "vgg16_correct"})
merged = r.merge(v, on="path", how="inner")

overlap_counts = pd.Series({
    "Both correct": int(((merged["resnet18_correct"] == 1) & (merged["vgg16_correct"] == 1)).sum()),
    "ResNet18 only correct": int(((merged["resnet18_correct"] == 1) & (merged["vgg16_correct"] == 0)).sum()),
    "VGG16 only correct": int(((merged["resnet18_correct"] == 0) & (merged["vgg16_correct"] == 1)).sum()),
    "Both wrong": int(((merged["resnet18_correct"] == 0) & (merged["vgg16_correct"] == 0)).sum()),
}).reset_index()
overlap_counts.columns = ["case", "count"]

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=overlap_counts, x="case", y="count", ax=ax, palette="Set3")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.set_xlabel("")
ax.set_ylabel("Number of test images")
ax.set_title("Prediction Agreement Between Models")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / "10_prediction_agreement.png")
plt.show()

overlap_counts

## Export Summary Table

In [ ]:
summary = comparison.copy()
summary["model"] = summary["model"].map(DISPLAY_NAMES)
summary["test_accuracy_pct"] = summary["test_accuracy"] * 100
summary["macro_f1_pct"] = summary["macro_f1"] * 100
summary["weighted_f1_pct"] = summary["weighted_f1"] * 100
summary["training_minutes"] = summary["training_seconds"] / 60
summary_export = summary[["model", "test_accuracy_pct", "macro_f1_pct", "weighted_f1_pct", "best_val_accuracy", "best_epoch", "training_minutes"]]
summary_export.to_csv(FIG_DIR / "model_summary_for_report.csv", index=False)
summary_export